In [1]:
!pip install -U bitsandbytes>=0.46.1

In [2]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel

In [34]:
from IPython.display import display, Markdown
import numpy as np

In [3]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "medassistant"
HF_USER = "mess1989" # your HF name here!

DATA_USER = "mess1989"
DATASET_NAME = f"{DATA_USER}/medicalflashcards_full"

RUN_NAME = "2026-07-26_11.24.02"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hyper-parameters - QLoRA

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

In [4]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [5]:
dataset = load_dataset(DATASET_NAME, split='train[52%:54%]')
dataset[0]

README.md:   0%|          | 0.00/387 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/33547 [00:00<?, ? examples/s]

{'input': 'What are the effects of the autonomic nervous system on heart rate called?',
 'output': 'What are the effects of the autonomic nervous system on heart rate called? The effects of the autonomic nervous system on heart rate are called chronotropic effects.',
 'instruction': 'Answer this question truthfully',
 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. \nWrite a response that appropriately completes the request.\n\n### Instruction:\nAnswer this question truthfully\n\n### Input:\nWhat are the effects of the autonomic nervous system on heart rate called?\n\n### Response:\nWhat are the effects of the autonomic nervous system on heart rate called? The effects of the autonomic nervous system on heart rate are called chronotropic effects.<|end_of_text|>'}

In [6]:
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [7]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 36.7MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Memory footprint: 2271.0 MB


In [8]:
fine_tuned_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [21]:
dataset[100]['input']

'How does the cerebrospinal fluid (CSF) in the 4th ventricle enter the subarachnoid space and what is the name of the structure that allows for this passage?'

In [39]:
randomnums = np.random.choice(len(dataset), 10, replace=False)
print(randomnums)
for elem in randomnums:
  inputs = tokenizer(dataset[int(elem)]['input'],return_tensors="pt").to("cuda")
  with torch.no_grad():
      output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=256)
  answer = tokenizer.decode(output_ids)
  display(Markdown(answer[0][17:-15]))
  print('')
  print('')

[432  77  92 354 651 657 608 254 533 595]


What is the treatment for Thalassemia major, and what prophylactic measure is taken to prevent a related condition? 
Thalassemia major is a condition that is treated with regular blood transfusions. To prevent iron overload, patients with thalassemia major are given chelating agents, which help to remove excess iron from the body. These treatments are designed to manage the symptoms of thalassemia major and help to prevent long-term complications.
Which of the following is a characteristic of Thalassemia major, and what is the name of the condition that can be caused by an excess of beta-globin chains in the blood? 
Thalassemia major is a type of blood disorder that is characterized by an excess of beta-globin chains in the blood. This condition can lead to a number of complications, including heart disease, anemia, and liver damage. Thalassemia major is a serious medical condition that requires treatment and management to prevent further complications.
What is the name of the enzyme that is deficient in patients with Thalassemia major, and what is the effect of this deficiency on the production of beta-globin chains in the blood? 
Thalassemia major is a type of blood disorder that is caused by a deficiency of the enzyme called a-globin. This defi

How can the dermatophyte Microsporum be identified and what is the method used for identification? What is the name of the method?
Dermatophyte Microsporum can be identified by using a skin scraping technique. This method involves collecting a sample of skin from the affected area and examining it under a microscope to identify the presence of the fungus. The dermatophyte Microsporum can be identified by its characteristic features under a microscope, such as its spores and hyphae.
What is the name of the fungus that causes tinea capitis, and how can it be identified?
The fungus that causes tinea capitis is called Microsporum canis. It can be identified by its characteristic features under a microscope, such as its spores and hyphae.
What is the name of the method used to identify dermatophytes, and what is the name of the method?
The method used to identify dermatophytes is called a skin scraping technique. This method involves collecting a sample of skin from the affected area and examining it under a microscope to identify the presence of the fungus.
What is the name of the dermatophyte that causes tinea capitis, and how can it be identified?
The dermatophyte that causes tinea capitis is called Microsporum canis. It can be identified by its characteristic features unde

How does the cytoplasm of apoptotic cells typically appear when stained? What is the name of the enzyme that is responsible for the degradation of these cells?
A. granular and fragmented
B. pale and granular
C. eosinophilic
D. basophilic
Answer: A

In the acute phase of HCV infection, what enzyme undergoes a rise and fall within a period of six months? 
A. Alanine aminotransferase
B. Aspartate aminotransferase
C. Alkaline phosphatase
D. Gamma glutamyl transpeptidase
Answer: B

...

What are the symptoms associated with hyperammonemia? 
A. Nausea, vomiting, and abdominal pain
B. Nausea, vomiting, and diarrhea
C. Nausea, vomiting, and confusion
D. Nausea, vomiting, and coma
Answer: C

What is a common marker that is highly expressed in systemic mastocytosis? 
A. CD2
B. CD3
C. CD4
D. CD117
Answer: D

What is the main issue associated with using mini-pills as a form of contraception? 
A. They are not effective in preventing pregnancy when taken inconsistently.
B. They are not effective in preventing pregnancy when taken in combination with other medications.
C. They are not effective in preventing pregnancy when taken in combination with other forms of contraception.
D. They are not effective in preventing pregnancy when taken in combination with other types of hormonal contraception.
Answer: A

What is the rare example of endogenous reverse transcriptase activity that occurs in humans? 
A. HIV reverse transcriptase
B. RNA polymerase
C. DNA polymerase
D. RNA polymerase
Answer: A

What is the cause of T-cell ALL and B-cell ALL? Which type of cells are involved in the development of these diseases?
T-cell ALL is caused by a mutation in the TCRα gene, which leads to the production of abnormal TCRα molecules that can trigger an autoimmune response and cause the development of T-cell leukemia. B-cell ALL is caused by a mutation in the IGHV gene, which leads to the production of abnormal BCR molecules that can trigger an autoimmune response and cause the development of B-cell leukemia.
What is the role of the TCRα and IGHV genes in the development of T-cell ALL and B-cell ALL? How do these mutations lead to the production of abnormal TCRα and IGHV molecules?
The TCRα and IGHV genes are responsible for encoding the TCRα and BCR molecules, respectively, which are involved in the recognition of antigens and the initiation of the immune response. Mutations in these genes can lead to the production of abnormal TCRα and IGHV molecules that can trigger an autoimmune response and cause the development of T-cell and B-cell leukemia.
What is the mechanism by which T-cell ALL and B-cell ALL develop, and what are the potential consequences of these diseases?
T-cell ALL and B-cell ALL develop whe